In [1]:
# Hamza Alsihati
# 2230006810

import numpy as np
import pandas as pd
import kagglehub
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [2]:
# Embedding: turns each word index into a dense vector that the model learns, so words with similar meanings get similar vectors
# Encoder LSTM: reads the English sentence word by word and compresses its meaning into one fixed-size vector
# Decoder LSTM: takes that vector, repeated once for each French position, and produces one output for every word of the translation
# Softmax: turns each output into probabilities over the French vocabulary, so the word with the highest probability is picked

In [3]:
path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")
dataframe = pd.read_csv(f"{path}/eng_-french.csv")
dataframe.columns = ["english", "french"]
print(len(dataframe))
dataframe = dataframe.head(20000)
dataframe.head()

175621


,english,french
0,Hi.,Salut!
1,Run!,Cours !
2,Run!,Courez !
3,Who?,Qui ?
4,Wow!,Ça alors !


In [4]:
tokenizer_eng = Tokenizer()
tokenizer_eng.fit_on_texts(dataframe["english"])
eng_padded = pad_sequences(tokenizer_eng.texts_to_sequences(dataframe["english"]), padding="post")

tokenizer_frn = Tokenizer()
tokenizer_frn.fit_on_texts(dataframe["french"])
frn_padded = pad_sequences(tokenizer_frn.texts_to_sequences(dataframe["french"]), padding="post")

print(eng_padded.shape, frn_padded.shape)
print(len(tokenizer_eng.word_index), len(tokenizer_frn.word_index))

(20000, 6) (20000, 12)
3514 7217


In [5]:
eng_train, eng_test, frn_train, frn_test = train_test_split(eng_padded, frn_padded, test_size=0.2, random_state=42)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(eng_padded.shape[1],)),
    tf.keras.layers.Embedding(input_dim=len(tokenizer_eng.word_index) + 1, output_dim=256),
    tf.keras.layers.LSTM(256),
    tf.keras.layers.RepeatVector(frn_padded.shape[1]),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(tokenizer_frn.word_index) + 1, activation="softmax"))
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 6, 256)         │       899,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 12, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 12, 256)        │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 12, 7218)       │     1,855,026 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,805,490 (14.52 MB)

 Trainable params: 3,805,490 (14.52 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(eng_train, np.expand_dims(frn_train, -1), epochs=30, batch_size=64, validation_split=0.2, verbose=2)

Epoch 1/30
200/200 - 8s - 41ms/step - accuracy: 0.6907 - loss: 2.7173 - val_accuracy: 0.7043 - val_loss: 2.2287
Epoch 2/30
200/200 - 7s - 34ms/step - accuracy: 0.7060 - loss: 2.1231 - val_accuracy: 0.7068 - val_loss: 2.1471
Epoch 3/30
200/200 - 7s - 36ms/step - accuracy: 0.7088 - loss: 2.0388 - val_accuracy: 0.7101 - val_loss: 2.1018
Epoch 4/30
200/200 - 7s - 36ms/step - accuracy: 0.7114 - loss: 1.9925 - val_accuracy: 0.7114 - val_loss: 2.0880
Epoch 5/30
200/200 - 7s - 36ms/step - accuracy: 0.7142 - loss: 1.9580 - val_accuracy: 0.7135 - val_loss: 2.0574
Epoch 6/30
200/200 - 7s - 36ms/step - accuracy: 0.7158 - loss: 1.9252 - val_accuracy: 0.7149 - val_loss: 2.0449
Epoch 7/30
200/200 - 7s - 36ms/step - accuracy: 0.7179 - loss: 1.8909 - val_accuracy: 0.7163 - val_loss: 2.0203
Epoch 8/30
200/200 - 7s - 37ms/step - accuracy: 0.7191 - loss: 1.8613 - val_accuracy: 0.7165 - val_loss: 2.0085
Epoch 9/30
200/200 - 7s - 36ms/step - accuracy: 0.7223 - loss: 1.8245 - val_accuracy: 0.7245 - val_loss:

In [7]:
loss, accuracy = model.evaluate(eng_test, np.expand_dims(frn_test, -1), verbose=0)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.7962291240692139


In [8]:
def translate_sentence(sentence):
    sequence = pad_sequences(tokenizer_eng.texts_to_sequences([sentence]), maxlen=eng_padded.shape[1], padding="post")
    prediction = model.predict(sequence, verbose=0)
    return tokenizer_frn.sequences_to_texts(np.argmax(prediction, axis=-1))[0]

for sentence in ["How are you?", "I like it.", "This is too expensive.", "Thank you very much.", "I want to buy this."]:
    print(sentence, "->", translate_sentence(sentence))

How are you? -> comment vous
I like it. -> j'aime voyager
This is too expensive. -> c'est trop trop
Thank you very much. -> jésus étais
I want to buy this. -> je veux le
